# EDA Cuaca Indonesia 2023–2024

Pada Milestone 3, saya memakai data bersih untuk melihat perubahan suhu dari waktu ke waktu dan perbedaan antarstasiun. Saya juga mengecek jumlah pengamatan dan data kosong karena keduanya memengaruhi cara membaca grafik.

Pertanyaan yang saya pakai:

1. Apakah jumlah laporan dan cakupan stasiun sama setiap bulan?
2. Bagaimana pola suhu bulanan dan pola suhu menurut jam pengamatan?
3. Bagaimana perbedaan suhu antarstasiun dan antara 2023 dengan 2024?

Hasilnya menggambarkan stasiun yang tersedia dalam dataset. Rata-rata ini bukan rata-rata suhu seluruh wilayah Indonesia, dan data dua tahun belum cukup untuk menyimpulkan tren iklim jangka panjang.


In [1]:
from pathlib import Path

import duckdb
import polars as pl
import plotly.express as px
import plotly.io as pio
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "noaa_indonesia_2023_2024.parquet"
FIGURE_DIR = PROJECT_ROOT / "output" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    raise FileNotFoundError("Jalankan notebook 02 untuk membuat data Parquet terlebih dahulu.")

pio.templates.default = "plotly_white"
pio.renderers.default = "plotly_mimetype"
con = duckdb.connect()
path_sql = DATA_PATH.as_posix().replace("'", "''")
con.execute(f"CREATE VIEW weather AS SELECT * FROM read_parquet('{path_sql}')")
figures = {}

def query(sql):
    return pl.from_arrow(con.execute(sql).to_arrow_table())

def show_chart(name, figure):
    figure.update_layout(margin=dict(l=60, r=40, t=90, b=65), font=dict(size=13), title_x=0.02)
    figure.write_html(FIGURE_DIR / f"{name}.html", include_plotlyjs="directory", full_html=True)
    figures[name] = figure
    figure.show()


In [2]:
summary = query("""SELECT COUNT(*) AS reports, COUNT(DISTINCT station_id) AS stations,
MIN(observed_at) AS first_time, MAX(observed_at) AS last_time FROM weather""")
print(summary)
assert summary["reports"][0] > 0
assert summary["stations"][0] > 1


shape: (1, 4)
┌─────────┬──────────┬─────────────────────┬─────────────────────┐
│ reports ┆ stations ┆ first_time          ┆ last_time           │
│ ---     ┆ ---      ┆ ---                 ┆ ---                 │
│ i64     ┆ i64      ┆ datetime[μs]        ┆ datetime[μs]        │
╞═════════╪══════════╪═════════════════════╪═════════════════════╡
│ 1273803 ┆ 114      ┆ 2023-01-01 00:00:00 ┆ 2024-12-31 23:30:00 │
└─────────┴──────────┴─────────────────────┴─────────────────────┘


## 1. Cek dulu cakupan datanya

Jumlah laporan belum tentu sama dengan jumlah waktu pengamatan yang berbeda. Saya melihat laporan per bulan terlebih dahulu, lalu menyiapkan analisis suhu agar stasiun yang lebih sering mengirim laporan tidak langsung mendapat bobot lebih besar.


In [3]:
coverage = query("""SELECT date_trunc('month', observed_at) AS month,
COUNT(*) AS reports, COUNT(DISTINCT station_id) AS stations,
COUNT(temperature_c) AS valid_temperature_reports
FROM weather GROUP BY month ORDER BY month""")
print(coverage)
assert coverage.height == 24


shape: (24, 4)
┌─────────────────────┬─────────┬──────────┬───────────────────────────┐
│ month               ┆ reports ┆ stations ┆ valid_temperature_reports │
│ ---                 ┆ ---     ┆ ---      ┆ ---                       │
│ datetime[μs]        ┆ i64     ┆ i64      ┆ i64                       │
╞═════════════════════╪═════════╪══════════╪═══════════════════════════╡
│ 2023-01-01 00:00:00 ┆ 49724   ┆ 114      ┆ 49695                     │
│ 2023-02-01 00:00:00 ┆ 44910   ┆ 114      ┆ 44874                     │
│ 2023-03-01 00:00:00 ┆ 50658   ┆ 114      ┆ 50605                     │
│ 2023-04-01 00:00:00 ┆ 50665   ┆ 114      ┆ 50629                     │
│ 2023-05-01 00:00:00 ┆ 51436   ┆ 114      ┆ 51397                     │
│ …                   ┆ …       ┆ …        ┆ …                         │
│ 2024-08-01 00:00:00 ┆ 49895   ┆ 114      ┆ 49865                     │
│ 2024-09-01 00:00:00 ┆ 71785   ┆ 114      ┆ 71649                     │
│ 2024-10-01 00:00:00 ┆ 73282   ┆ 11

In [4]:
fig = px.bar(coverage.to_pandas(), x="month", y="reports", color="stations",
hover_data=["valid_temperature_reports"], color_continuous_scale="Teal",
title="1. Jumlah laporan per bulan",
labels={"month": "Bulan", "reports": "Jumlah laporan", "stations": "Stasiun aktif", "valid_temperature_reports": "Laporan dengan suhu valid"})
show_chart("01_monthly_coverage", fig)


In [5]:
variables = ["temperature_c", "dew_point_c", "sea_level_pressure_hpa", "wind_speed_ms", "visibility_m", "ceiling_m"]
missing = pl.DataFrame([
    {"variable": variable, "missing_pct": con.execute(f"SELECT 100.0 * (COUNT(*) - COUNT({variable})) / COUNT(*) FROM weather").fetchone()[0]}
    for variable in variables
]).sort("missing_pct", descending=True)
print(missing)


shape: (6, 2)
┌────────────────────────┬─────────────┐
│ variable               ┆ missing_pct │
│ ---                    ┆ ---         │
│ str                    ┆ f64         │
╞════════════════════════╪═════════════╡
│ ceiling_m              ┆ 63.066973   │
│ sea_level_pressure_hpa ┆ 49.558605   │
│ wind_speed_ms          ┆ 0.252551    │
│ dew_point_c            ┆ 0.182838    │
│ visibility_m           ┆ 0.177735    │
│ temperature_c          ┆ 0.146883    │
└────────────────────────┴─────────────┘


In [6]:
fig = px.bar(missing.to_pandas(), x="missing_pct", y="variable", orientation="h",
text="missing_pct", title="2. Data kosong setelah cleaning",
labels={"variable": "Variabel", "missing_pct": "Data kosong (%)"})
fig.update_traces(texttemplate="%{text:.2f}%", marker_color="#d97706")
fig.update_layout(yaxis=dict(categoryorder="total ascending"))
show_chart("02_missing_values", fig)


## 2. Siapkan suhu yang lebih sebanding

Saya merata-ratakan laporan pada stasiun dan waktu yang sama, lalu menggabungkannya menjadi satu nilai per stasiun per jam. Setelah itu, saya menghitung rata-rata harian dan rata-rata bulanan per stasiun. Dengan cara ini, banyaknya laporan dalam satu jam tidak langsung mendominasi rata-rata bulanan.

Untuk membandingkan bulan, saya memakai stasiun dengan setidaknya 20 hari bersuhu valid pada setiap bulan selama 24 bulan. Setiap stasiun yang lolos mendapat bobot sama. Batas 20 hari adalah pilihan analisis, bukan jaminan seluruh jam lengkap. Waktu tetap mengikuti UTC pada data NOAA, bukan WIB, WITA, atau WIT.


In [7]:
con.execute("""CREATE TEMP TABLE temperature_times AS
SELECT station_id, observed_at, AVG(temperature_c) AS temperature_c
FROM weather WHERE temperature_c IS NOT NULL GROUP BY station_id, observed_at""")
con.execute("""CREATE TEMP TABLE hourly AS
SELECT station_id, date_trunc('hour', observed_at) AS hour_utc,
AVG(temperature_c) AS temperature_c
FROM temperature_times GROUP BY station_id, hour_utc""")
print(query("SELECT COUNT(*) AS station_hours FROM hourly"))


shape: (1, 1)
┌───────────────┐
│ station_hours │
│ ---           │
│ i64           │
╞═══════════════╡
│ 863231        │
└───────────────┘


In [8]:
con.execute("""CREATE TEMP TABLE daily AS
SELECT station_id, CAST(hour_utc AS DATE) AS day,
AVG(temperature_c) AS temperature_c, COUNT(*) AS valid_hours
FROM hourly GROUP BY station_id, day""")
con.execute("""CREATE TEMP TABLE station_month AS
SELECT station_id, date_trunc('month', day) AS month,
AVG(temperature_c) AS temperature_c, COUNT(*) AS valid_days,
AVG(valid_hours) AS mean_hours_per_day
FROM daily GROUP BY station_id, month""")
print(query("SELECT MIN(valid_days) AS min_days, MAX(valid_days) AS max_days, MIN(mean_hours_per_day) AS min_mean_hours FROM station_month"))


shape: (1, 3)
┌──────────┬──────────┬────────────────┐
│ min_days ┆ max_days ┆ min_mean_hours │
│ ---      ┆ ---      ┆ ---            │
│ i64      ┆ i64      ┆ f64            │
╞══════════╪══════════╪════════════════╡
│ 5        ┆ 31       ┆ 1.142857       │
└──────────┴──────────┴────────────────┘


In [9]:
MIN_DAYS = 20
con.execute(f"""CREATE TEMP TABLE panel AS
SELECT station_id FROM station_month
GROUP BY station_id HAVING COUNT(*) = 24 AND MIN(valid_days) >= {MIN_DAYS}""")
panel_count = con.execute("SELECT COUNT(*) FROM panel").fetchone()[0]
assert panel_count > 0, "Tidak ada stasiun yang memenuhi syarat. Tinjau cakupan sebelum melanjutkan."
print(f"Stasiun yang dipakai: {panel_count} dari {summary['stations'][0]}")
print(query("""SELECT MIN(valid_days) AS min_days,
MIN(mean_hours_per_day) AS min_mean_hours, MAX(mean_hours_per_day) AS max_mean_hours
FROM station_month JOIN panel USING(station_id)"""))


Stasiun yang dipakai: 50 dari 114
shape: (1, 3)
┌──────────┬────────────────┬────────────────┐
│ min_days ┆ min_mean_hours ┆ max_mean_hours │
│ ---      ┆ ---            ┆ ---            │
│ i64      ┆ f64            ┆ f64            │
╞══════════╪════════════════╪════════════════╡
│ 20       ┆ 3.433333       ┆ 24.0           │
└──────────┴────────────────┴────────────────┘


In [10]:
monthly = query("""SELECT month, AVG(temperature_c) AS temperature_c,
MIN(temperature_c) AS lowest_station_mean, MAX(temperature_c) AS highest_station_mean,
COUNT(*) AS stations FROM station_month JOIN panel USING(station_id)
GROUP BY month ORDER BY month""")
assert monthly.height == 24
assert monthly["stations"].n_unique() == 1
print(monthly)


shape: (24, 5)
┌─────────────────────┬───────────────┬─────────────────────┬──────────────────────┬──────────┐
│ month               ┆ temperature_c ┆ lowest_station_mean ┆ highest_station_mean ┆ stations │
│ ---                 ┆ ---           ┆ ---                 ┆ ---                  ┆ ---      │
│ datetime[μs]        ┆ f64           ┆ f64                 ┆ f64                  ┆ i64      │
╞═════════════════════╪═══════════════╪═════════════════════╪══════════════════════╪══════════╡
│ 2023-01-01 00:00:00 ┆ 27.168234     ┆ 21.044355           ┆ 29.035454            ┆ 50       │
│ 2023-02-01 00:00:00 ┆ 27.048208     ┆ 20.555038           ┆ 28.641071            ┆ 50       │
│ 2023-03-01 00:00:00 ┆ 27.345425     ┆ 21.262043           ┆ 28.608863            ┆ 50       │
│ 2023-04-01 00:00:00 ┆ 27.783789     ┆ 22.180833           ┆ 29.230111            ┆ 50       │
│ 2023-05-01 00:00:00 ┆ 28.032492     ┆ 22.42644            ┆ 29.542473            ┆ 50       │
│ …                   ┆ …

In [11]:
fig = px.line(monthly.to_pandas(), x="month", y="temperature_c", markers=True,
hover_data=["stations", "lowest_station_mean", "highest_station_mean"],
title=f"3. Rata-rata suhu bulanan pada {panel_count} stasiun yang sama",
labels={"month": "Bulan", "temperature_c": "Suhu rata-rata (°C)", "stations": "Jumlah stasiun", "lowest_station_mean": "Rata-rata stasiun terendah", "highest_station_mean": "Rata-rata stasiun tertinggi"})
fig.update_traces(line_color="#0f766e")
show_chart("03_monthly_temperature", fig)


## 3. Lihat pola antarstasiun dan jam pengamatan

Grafik rata-rata bisa menyembunyikan perbedaan antarstasiun. Karena itu, saya menampilkan suhu bulanan setiap stasiun yang lolos syarat cakupan. Label memakai ID agar nama yang sama tidak dianggap sebagai satu stasiun.

Untuk pola jam, rata-rata dihitung per stasiun dan jam UTC terlebih dahulu, baru dirata-ratakan antarstasiun. Jumlah stasiun yang tersedia pada tiap jam ikut ditampilkan.


In [12]:
station_matrix = query("""SELECT station_id, strftime(month, '%Y-%m') AS month,
temperature_c FROM station_month JOIN panel USING(station_id)
ORDER BY station_id, month""").to_pandas().pivot(index="station_id", columns="month", values="temperature_c")
assert not station_matrix.isna().any().any()
print(f"Matriks suhu: {station_matrix.shape}")


Matriks suhu: (50, 24)


In [13]:
fig = px.imshow(station_matrix, aspect="auto", color_continuous_scale="YlOrRd",
title="4. Suhu bulanan tiap stasiun dalam kelompok yang sama",
labels={"x": "Bulan", "y": "ID stasiun", "color": "Suhu (°C)"})
fig.update_layout(height=max(650, panel_count * 13))
show_chart("04_station_month_heatmap", fig)


In [14]:
diurnal = query("""WITH station_hour AS (
SELECT station_id, EXTRACT(hour FROM hour_utc)::INTEGER AS utc_hour,
AVG(temperature_c) AS temperature_c, COUNT(*) AS observations
FROM hourly JOIN panel USING(station_id) GROUP BY station_id, utc_hour)
SELECT utc_hour, AVG(temperature_c) AS temperature_c, COUNT(*) AS stations
FROM station_hour GROUP BY utc_hour ORDER BY utc_hour""")
print(diurnal)


shape: (24, 3)
┌──────────┬───────────────┬──────────┐
│ utc_hour ┆ temperature_c ┆ stations │
│ ---      ┆ ---           ┆ ---      │
│ i32      ┆ f64           ┆ i64      │
╞══════════╪═══════════════╪══════════╡
│ 0        ┆ 26.783175     ┆ 50       │
│ 1        ┆ 28.674123     ┆ 35       │
│ 2        ┆ 29.7316       ┆ 35       │
│ 3        ┆ 30.110876     ┆ 50       │
│ 4        ┆ 30.904063     ┆ 35       │
│ …        ┆ …             ┆ …        │
│ 19       ┆ 25.493195     ┆ 35       │
│ 20       ┆ 25.31266      ┆ 35       │
│ 21       ┆ 25.079812     ┆ 48       │
│ 22       ┆ 25.350761     ┆ 35       │
│ 23       ┆ 26.087033     ┆ 35       │
└──────────┴───────────────┴──────────┘


In [15]:
fig = px.line(diurnal.to_pandas(), x="utc_hour", y="temperature_c", markers=True,
hover_data=["stations"], title="5. Pola suhu menurut jam UTC",
labels={"utc_hour": "Jam UTC", "temperature_c": "Suhu rata-rata (°C)", "stations": "Stasiun yang tersedia"})
fig.update_xaxes(dtick=1)
show_chart("05_diurnal_temperature_utc", fig)


## 4. Bandingkan lokasi dan tahun pengamatan

Diagram lokasi memakai bujur dan lintang tanpa peta dasar. Warna titik menunjukkan rata-rata dari 24 suhu bulanan setiap stasiun. Grafik ini membantu melihat pola lokasi, tetapi belum membuktikan penyebab perbedaannya; elevasi dan cakupan pengamatan juga bisa berpengaruh.

Perbandingan tahun memakai stasiun yang sama dan bobot yang sama untuk setiap bulan. Selisih 2024 dikurangi 2023 hanya menjelaskan dua tahun dalam dataset ini.


In [16]:
locations = query("""WITH metadata AS (
SELECT station_id, MIN(station_name) AS station_name, MEDIAN(latitude) AS latitude,
MEDIAN(longitude) AS longitude, MEDIAN(elevation_m) AS elevation_m
FROM weather GROUP BY station_id)
SELECT station_id, station_name, latitude, longitude, elevation_m,
AVG(temperature_c) AS temperature_c
FROM station_month JOIN panel USING(station_id) JOIN metadata USING(station_id)
GROUP BY station_id, station_name, latitude, longitude, elevation_m
ORDER BY station_id""")
print(locations.head(10))


shape: (10, 6)
┌─────────────┬─────────────────────────────┬───────────┬────────────┬─────────────┬───────────────┐
│ station_id  ┆ station_name                ┆ latitude  ┆ longitude  ┆ elevation_m ┆ temperature_c │
│ ---         ┆ ---                         ┆ ---       ┆ ---        ┆ ---         ┆ ---           │
│ str         ┆ str                         ┆ f64       ┆ f64        ┆ f64         ┆ f64           │
╞═════════════╪═════════════════════════════╪═══════════╪════════════╪═════════════╪═══════════════╡
│ 96001099999 ┆ MAIMUN SALEH, ID            ┆ 5.874131  ┆ 95.339672  ┆ 113.0       ┆ 27.80065      │
│ 96015099999 ┆ MEULABOH CUT NYAK, ID       ┆ 4.25      ┆ 96.116667  ┆ 90.0        ┆ 27.426409     │
│ 96033099999 ┆ MEDAN BELAWAN, ID           ┆ 3.8       ┆ 98.7       ┆ 3.0         ┆ 28.599193     │
│ 96035099999 ┆ POLONIA, ID                 ┆ 3.63787   ┆ 98.87053   ┆ 7.0         ┆ 27.58653      │
│ 96087099999 ┆ BATAM HANG NADIM, ID        ┆ 1.1166666 ┆ 104.116667 ┆ 24.0 

In [17]:
fig = px.scatter(locations.to_pandas(), x="longitude", y="latitude", color="temperature_c",
hover_name="station_name", hover_data=["station_id", "elevation_m"],
color_continuous_scale="YlOrRd", title="6. Sebaran suhu stasiun berdasarkan koordinat",
labels={"longitude": "Bujur (°)", "latitude": "Lintang (°)", "temperature_c": "Suhu (°C)", "station_id": "ID stasiun", "elevation_m": "Elevasi (m)"})
fig.update_traces(marker_size=11, marker_line_width=0.5, marker_line_color="#333333")
fig.update_yaxes(scaleanchor="x", scaleratio=1)
show_chart("06_station_locations", fig)


In [18]:
year_comparison = query("""WITH yearly AS (
SELECT station_id, EXTRACT(year FROM month)::INTEGER AS year,
AVG(temperature_c) AS temperature_c FROM station_month JOIN panel USING(station_id)
GROUP BY station_id, year)
SELECT station_id,
MAX(CASE WHEN year=2023 THEN temperature_c END) AS temperature_2023,
MAX(CASE WHEN year=2024 THEN temperature_c END) AS temperature_2024,
MAX(CASE WHEN year=2024 THEN temperature_c END) -
MAX(CASE WHEN year=2023 THEN temperature_c END) AS difference_c
FROM yearly GROUP BY station_id ORDER BY difference_c""")
print(year_comparison)


shape: (50, 4)
┌─────────────┬──────────────────┬──────────────────┬──────────────┐
│ station_id  ┆ temperature_2023 ┆ temperature_2024 ┆ difference_c │
│ ---         ┆ ---              ┆ ---              ┆ ---          │
│ str         ┆ f64              ┆ f64              ┆ f64          │
╞═════════════╪══════════════════╪══════════════════╪══════════════╡
│ 97682099999 ┆ 28.746466        ┆ 28.19865         ┆ -0.547816    │
│ 96749599999 ┆ 29.190731        ┆ 28.834594        ┆ -0.356137    │
│ 96295099999 ┆ 27.986286        ┆ 27.706912        ┆ -0.279374    │
│ 97014099999 ┆ 27.209895        ┆ 27.020064        ┆ -0.189831    │
│ 96633099999 ┆ 27.84274         ┆ 27.764179        ┆ -0.078561    │
│ …           ┆ …                ┆ …                ┆ …            │
│ 97372599999 ┆ 28.269445        ┆ 28.866158        ┆ 0.596713     │
│ 97372099999 ┆ 27.205269        ┆ 27.806227        ┆ 0.600958     │
│ 96805099999 ┆ 27.213402        ┆ 27.830656        ┆ 0.617254     │
│ 97230099999 ┆ 27.

In [19]:
fig = px.bar(year_comparison.to_pandas(), x="station_id", y="difference_c",
color="difference_c", color_continuous_scale="RdBu_r", color_continuous_midpoint=0,
hover_data=["temperature_2023", "temperature_2024"],
title="7. Selisih suhu rata-rata 2024 terhadap 2023 per stasiun",
labels={"station_id": "ID stasiun", "difference_c": "Selisih suhu (°C)", "temperature_2023": "Suhu 2023 (°C)", "temperature_2024": "Suhu 2024 (°C)"})
fig.update_xaxes(type="category")
show_chart("07_yearly_temperature_difference", fig)


## 5. Catatan dari hasil EDA

Angka berikut dihitung langsung dari tabel analisis agar bisa diperbarui saat data berubah. Saya membedakan temuan tentang kelengkapan data dari temuan tentang pola suhu.


In [20]:
warmest = monthly.sort("temperature_c", descending=True).row(0, named=True)
coolest = monthly.sort("temperature_c").row(0, named=True)
peak_hour = diurnal.sort("temperature_c", descending=True).row(0, named=True)
low_hour = diurnal.sort("temperature_c").row(0, named=True)
warmest_station = locations.sort("temperature_c", descending=True).row(0, named=True)
coolest_station = locations.sort("temperature_c").row(0, named=True)
mean_difference = year_comparison["difference_c"].mean()
warmer_stations = year_comparison.filter(pl.col("difference_c") > 0).height
largest_gap = missing.row(0, named=True)
insights = [
    f"Data mencakup {summary['reports'][0]:,} laporan dari {summary['stations'][0]} stasiun. Dengan syarat minimal {MIN_DAYS} hari valid per bulan, {panel_count} stasiun dipakai untuk perbandingan 24 bulan.",
    f"Pada kelompok stasiun yang sama, suhu bulanan tertinggi ada pada {warmest['month']:%Y-%m} ({warmest['temperature_c']:.2f} °C), sedangkan yang terendah pada {coolest['month']:%Y-%m} ({coolest['temperature_c']:.2f} °C).",
    f"Rata-rata selisih suhu 2024 terhadap 2023 adalah {mean_difference:+.2f} °C. Sebanyak {warmer_stations} dari {panel_count} stasiun memiliki selisih positif. Perbandingan memakai bobot bulan dan stasiun yang sama.",
    f"Pada kelompok jam UTC, suhu tertinggi muncul pada jam {peak_hour['utc_hour']:02d} ({peak_hour['temperature_c']:.2f} °C) dan terendah pada jam {low_hour['utc_hour']:02d} ({low_hour['temperature_c']:.2f} °C). Jam ini tidak boleh langsung disebut jam lokal Indonesia.",
    f"Rata-rata 24 bulan tertinggi ada pada {warmest_station['station_name']} ({warmest_station['station_id']}, {warmest_station['temperature_c']:.2f} °C), dan terendah pada {coolest_station['station_name']} ({coolest_station['station_id']}, {coolest_station['temperature_c']:.2f} °C). Perbedaan lokasi dan elevasi perlu dipertimbangkan sebelum menjelaskan penyebabnya.",
    f"Variabel dengan data kosong terbesar adalah {largest_gap['variable']} ({largest_gap['missing_pct']:.2f}%). Analisis variabel ini perlu memeriksa cakupannya terlebih dahulu.",
]
display(Markdown(chr(10).join(f"{i}. {text}" for i, text in enumerate(insights, 1))))


1. Data mencakup 1,273,803 laporan dari 114 stasiun. Dengan syarat minimal 20 hari valid per bulan, 50 stasiun dipakai untuk perbandingan 24 bulan.
2. Pada kelompok stasiun yang sama, suhu bulanan tertinggi ada pada 2023-11 (28.36 °C), sedangkan yang terendah pada 2023-02 (27.05 °C).
3. Rata-rata selisih suhu 2024 terhadap 2023 adalah +0.23 °C. Sebanyak 42 dari 50 stasiun memiliki selisih positif. Perbandingan memakai bobot bulan dan stasiun yang sama.
4. Pada kelompok jam UTC, suhu tertinggi muncul pada jam 05 (31.14 °C) dan terendah pada jam 21 (25.08 °C). Jam ini tidak boleh langsung disebut jam lokal Indonesia.
5. Rata-rata 24 bulan tertinggi ada pada SURABAYA GEDANGAN, ID (96937099999, 29.36 °C), dan terendah pada BOGOR CITEKO, ID (96751099999, 22.00 °C). Perbedaan lokasi dan elevasi perlu dipertimbangkan sebelum menjelaskan penyebabnya.
6. Variabel dengan data kosong terbesar adalah ceiling_m (63.07%). Analisis variabel ini perlu memeriksa cakupannya terlebih dahulu.

In [21]:
assert len(figures) == 7
assert year_comparison.height == panel_count
assert year_comparison.null_count().select(pl.sum_horizontal(pl.all())).item() == 0
assert all(len(fig.data) > 0 for fig in figures.values())
assert all((FIGURE_DIR / f"{name}.html").stat().st_size > 1000 for name in figures)
for name in figures:
    print(FIGURE_DIR / f"{name}.html")
print("Tujuh grafik interaktif berhasil dibuat dan tabel perbandingan lengkap.")


/home/jovyan/work/output/figures/01_monthly_coverage.html
/home/jovyan/work/output/figures/02_missing_values.html
/home/jovyan/work/output/figures/03_monthly_temperature.html
/home/jovyan/work/output/figures/04_station_month_heatmap.html
/home/jovyan/work/output/figures/05_diurnal_temperature_utc.html
/home/jovyan/work/output/figures/06_station_locations.html
/home/jovyan/work/output/figures/07_yearly_temperature_difference.html
Tujuh grafik interaktif berhasil dibuat dan tabel perbandingan lengkap.


## Batasan yang masih perlu diperhatikan

Tidak semua stasiun memiliki jumlah jam valid yang sama. Rata-rata harian dihitung dari jam yang tersedia, sehingga hasil bisa dipengaruhi jam yang hilang. Grafik menurut jam menampilkan jumlah stasiun yang tersedia pada setiap jam.

Syarat 20 hari per bulan membantu menyamakan kelompok stasiun, tetapi mengurangi jumlah stasiun yang dipakai. Hasil tidak mewakili seluruh daratan Indonesia secara merata. Saya belum menguji signifikansi statistik atau meneliti penyebab perubahan suhu.

Grafik bisa dibuka dari output notebook di JupyterLab atau file HTML dalam output/figures. File HTML memakai plotly.min.js di folder yang sama, sehingga folder tersebut perlu tetap bersama saat dipindahkan. GitHub dapat menampilkan output notebook secara berbeda; file HTML bisa dibuat ulang dengan menjalankan notebook ini.
